In [3]:
import pandas as pd
from pathlib import Path

# Définissez le chemin vers votre fichier CSV
PROJECT_ROOT = Path(".").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CSV_PATH = PROCESSED_DIR / "kialo_pairs.csv"

# Chargez le CSV
df_pairs = pd.read_csv(CSV_PATH)

# On affiche la distribution des classes pour eviter une surepresentation d'une classe
print("\n Distribution des classes")
print(df_pairs["label"].value_counts())
print("\n Pourcentage des classes ")
print(df_pairs["label"].value_counts(normalize=True) * 100)


 Distribution des classes
label
No-Relation    96445
Attack         48689
Support        47810
Name: count, dtype: int64

 Pourcentage des classes 
label
No-Relation    49.986006
Attack         25.234783
Support        24.779211
Name: proportion, dtype: float64


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

# On va faire un split en s'assurant que les arguments d'un meme debat reste dans le meme split
unique_debates = df_pairs["debate_id"].unique()
print(f"Nombre total de débats uniques : {len(unique_debates)}")

# On decide de split en 70, 15, 15
train_debate, rest = train_test_split(unique_debates, test_size=0.3, random_state=42)
test_debate, val_debate = train_test_split(unique_debates, test_size=0.5, random_state=42)

train_df = df_pairs[df_pairs["debate_id"].isin(train_debate)]
val_df = df_pairs[df_pairs["debate_id"].isin(val_debate)]
test_df = df_pairs[df_pairs["debate_id"].isin(test_debate)]

print(f"Taille du Train : {len(train_df)} paires")
print(f"Taille du Val : {len(val_df)} paires")
print(f"Taille du Test  : {len(test_df)} paires")

Nombre total de débats uniques : 244
Taille du Train : 125188 paires
Taille du Val : 102069 paires
Taille du Test  : 90875 paires


In [5]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Mapping des labels en entiers (le modèle a besoin de chiffres, pas de texte)
label_map = {"No-Relation": 0, "Attack": 1, "Support": 2}
train_df["label_id"] = train_df["label"].map(label_map)
val_df["label_id"] = val_df["label"].map(label_map)
test_df["label_id"] = test_df["label"].map(label_map)

# On convertit nos df en dataset
cols_to_keep = ["arg1", "arg2", "label_id"]
hg_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[cols_to_keep], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[cols_to_keep], preserve_index=False),
    "test": Dataset.from_pandas(test_df[cols_to_keep], preserve_index=False)
})


MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization pour traduire du texte en token
def tokenize_function(examples):
    # RoBERTa gère les paires de phrases nativement si on lui passe deux listes
    # Il va automatiquement ajouter les tokens <s> arg1 </s></s> arg2 </s>
    return tokenizer(
        examples["arg1"],
        examples["arg2"],
        padding=True,
        truncation=True
    )

tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)

# On renomme label_id en labels (format attendu par le trainer)
tokenized_datasets = tokenized_datasets.rename_column("label_id", "labels")

print(tokenized_datasets)

/tmp/ipykernel_1241/749353441.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["label_id"] = train_df["label"].map(label_map)
/tmp/ipykernel_1241/749353441.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["label_id"] = val_df["label"].map(label_map)
/tmp/ipykernel_1241/749353441.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.py

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/125188 [00:00<?, ? examples/s]

Map:   0%|          | 0/102069 [00:00<?, ? examples/s]

Map:   0%|          | 0/90875 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['arg1', 'arg2', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 125188
    })
    validation: Dataset({
        features: ['arg1', 'arg2', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 102069
    })
    test: Dataset({
        features: ['arg1', 'arg2', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 90875
    })
})


In [6]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

# On charge le modèle RoBERTa

# num_labels=3 car nous avons [0: No-Relation, 1: Attack, 2: Support]
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# On prépare la métrique d'évaluation (Macro F1-Score)
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Le modèle sort des probabilités (logits), on prend la plus haute (argmax)
    predictions = np.argmax(logits, axis=-1)
    return metric_f1.compute(predictions=predictions, references=labels, average="macro")

# Pour aller plus vite
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(2000))

# Paramètres d'entraînement
training_args = TrainingArguments(
    output_dir= str(PROJECT_ROOT / "models" / "kialo_roberta_small"), # Dossier de sauvegarde
    eval_strategy="epoch",     # Evaluer à la fin de chaque itération
    save_strategy="epoch",           # Sauvegarder à la fin de chaque itération
    learning_rate=2e-5,              # Vitesse d'apprentissage classique pour RoBERTa
    per_device_train_batch_size=16,  # Nombre d'exemples lus en même temps
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # Le modèle va lire le dataset 3 fois en entier
    weight_decay=0.01,               # Evite le sur-apprentissage
    load_best_model_at_end=True,     # Garde le meilleur modèle à la fin
)

# Parce que le modele ne sait pas gérer les phrases de differentes tailles
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Initialisation du trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
print("Début de l'entraînement...")
trainer.train()

Début de l'entraînement...


Epoch,Training Loss,Validation Loss,F1
1,0.840126,0.694779,0.459466
2,0.607756,0.603323,0.719157
3,0.481182,0.606381,0.741745


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.5939846598307291, metrics={'train_runtime': 3352.4462, 'train_samples_per_second': 8.949, 'train_steps_per_second': 0.559, 'total_flos': 7893402531840000.0, 'train_loss': 0.5939846598307291, 'epoch': 3.0})